# Day 5 – AI Brochure Generator

This notebook builds an AI-powered brochure generator for startups.

The system:

1. Scrapes links from a company website
2. Uses an LLM to filter investor-relevant pages
3. Generates a structured startup brochure
4. Optionally translates the brochure to another language
5. Streams output for an interactive experience

All outputs are cleared before submission.

## Architecture Overview

The system follows a modular pipeline:

Website  
↓  
Link Scraping  
↓  
LLM Link Filtering  
↓  
Brochure Generation  
↓  
Optional Translation

Each stage is implemented as a separate function to keep the system modular and easy to extend.


## Install Dependencies

We install the required libraries for:

- Web scraping
- HTML parsing
- LLM API calls

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv(find_dotenv())

# Initialize OpenAI client
client = OpenAI()

# Model to use
MODEL = "gpt-5-nano"

## User Inputs

The user provides:

- Website URL
- Optional translation language

In [ ]:
site_url = "https://edwarddonner.com/"

target_language = "Spanish"   # None for English output

## Step 1 – Scrape Website Links

This function:

- Fetches the webpage
- Extracts all anchor tags
- Converts relative URLs to absolute URLs

In [ ]:
def scrape_links(base_url):

    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, "html.parser")

    links = set()

    for tag in soup.find_all("a", href=True):

        href = tag["href"]
        absolute = urljoin(base_url, href)

        if absolute.startswith("http"):
            links.add(absolute)

    return list(links)

## Step 2 – Filter Relevant Links Using LLM

Many website links are not useful for an investor brochure.

We use an LLM to keep only relevant links such as:

- Product pages
- Technology overview
- Company overview
- Case studies

In [ ]:
LINK_FILTER_SYSTEM_PROMPT = """
You are an AI assistant helping create an investor brochure.

From a list of website links, return ONLY links useful for understanding the company.

Include:
- product pages
- technology explanation
- company overview
- pricing
- solutions
- case studies

Exclude:
- privacy policy
- terms
- cookies
- login
- careers
- legal pages

Return JSON:

{
 "relevant_links":[
   {"url":"...","reason":"..."}
 ]
}
"""

In [ ]:
def filter_links(links):

    user_prompt = f"""
Analyze the following links and return only relevant ones.

Links:
{json.dumps(links, indent=2)}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":LINK_FILTER_SYSTEM_PROMPT},
            {"role":"user","content":user_prompt}
        ]
    )

    return response.choices[0].message.content

## Step 3 – Generate Investor Brochure

Using the filtered links, the model generates a structured brochure
for startup investors.

In [ ]:
BROCHURE_SYSTEM_PROMPT = """
You are an expert startup analyst.

Generate a professional investor brochure.

Include sections:

Product Overview
Target Market
Business Model
Technology
Key Advantages

Use concise professional language.
"""

## Streaming Output

To make the notebook interactive, the response is streamed token-by-token.

In [ ]:
def stream_response(system_prompt, user_prompt):

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
        ],
        stream=True
    )

    output=""

    for chunk in stream:

        token = chunk.choices[0].delta.content

        if token:
            print(token, end="", flush=True)
            output += token

    print("\n")

    return output

In [ ]:
def generate_brochure(filtered_links):

    user_prompt = f"""
Generate an investor brochure using the following company links:

{filtered_links}
"""

    return stream_response(
        BROCHURE_SYSTEM_PROMPT,
        user_prompt
    )

## Step 4 – Translate Brochure

The generated brochure can be translated for international investors.

In [ ]:
TRANSLATION_SYSTEM_PROMPT = """
You are a professional business translator.

Translate the brochure into the requested language.

Preserve formatting and tone.
"""

In [ ]:
def translate_brochure(text, language):

    user_prompt = f"""
Translate the following brochure into {language}:

{text}
"""

    return stream_response(
        TRANSLATION_SYSTEM_PROMPT,
        user_prompt
    )

## Step 5 – Run Full Pipeline

In [ ]:
links = scrape_links(site_url)

filtered_links = filter_links(links)

brochure = generate_brochure(filtered_links)

if target_language:
    brochure = translate_brochure(brochure, target_language)
from IPython.display import Markdown, display
display(Markdown(brochure))

html_content = f"<html><body><pre>{brochure}</pre></body></html>"

with open("brochure_output.html", "w", encoding="utf-8") as f:
    f.write(html_content)
